# TCC — Colab: treinamento modular YOLO26

**Altere somente a célula 1 (`PARAMS`) para configurar o notebook.** As outras células usam esses parâmetros.

Fluxo: dataset amplo → Modelo Geral → LoRAs Especialistas independentes → controles Full Fine-Tune.


## 1. PARÂMETROS GLOBAIS — configure tudo aqui


In [ ]:
from pathlib import Path

PARAMS = {
    "paths": {
        "repo": "/content/TCC_Code_Test",
        "drive_runtime": "/content/drive/MyDrive/TCC_Code_Test_runtime",
        "yolo_master": "/content/YOLO-Master",
        "local_data_root": "/content/TCC_training_data",
        "training_project": "workspace/training_runs",
        "analysis_export": "exports/TCC_Analise_models",
    },
    "secrets": {
        "github": "GITHUB_TOKEN",
        "roboflow": "ROBOFLOW_API_KEY",
        "ngrok": "NGROK_AUTHTOKEN",
    },
    "roboflow": {
        "workspace": "rafael-fernando-dos-reis-mecabo",
        "format": "yolo26",
        "general": {
            "project": "detecao-de-cetaceos-afrzs-iqey9",
            "version": 1,
            "output": "data/roboflow",
            "download": True,
            "overwrite": False,
        },
        "lora": {
            "det_lora_01": {"project": "COLOQUE_SLUG_LORA_01", "version": 1, "output": "data/specialists/detection/lora_01", "download": False},
            "det_lora_02": {"project": "COLOQUE_SLUG_LORA_02", "version": 1, "output": "data/specialists/detection/lora_02", "download": False},
            "det_lora_03": {"project": "COLOQUE_SLUG_LORA_03", "version": 1, "output": "data/specialists/detection/lora_03", "download": False},
            "det_lora_04": {"project": "COLOQUE_SLUG_LORA_04", "version": 1, "output": "data/specialists/detection/lora_04", "download": False},
            "det_lora_05": {"project": "COLOQUE_SLUG_LORA_05", "version": 1, "output": "data/specialists/detection/lora_05", "download": False}
        },
    },
    "training": {
        "detect_model": "yolo26n.pt",
        "segment_model": "yolo26n-seg.pt",
        "epochs": 100,
        "imgsz": 640,
        "batch": 32,
        "workers": 4,
        "device": 0,
        "amp": True,
        "optimizer": "auto",
        "patience": 30,
        "seed": 42,
        "deterministic": True,
        "cache": "disk",
        "close_mosaic": 10,
        "save_period": 10,
        "plots": True,
        "verbose": True,
        "eval_split": "test",
        "use_local_dataset_cache": True,
        "refresh_local_dataset_cache": False
    },
    "smoke": {"epochs": 3, "imgsz": 640, "batch": 32, "workers": 4, "cache": "disk"},
    "lora": {"r": 16, "alpha": 32, "dropout": 0.10, "gradient_checkpointing": True, "include_head": True},
    "clustering": {
        "enabled": False,
        "source": "data/roboflow/train",
        "output": "workspace/bbox_clusters_colab",
        "context_scale": 1.8,
        "min_crop_size_px": 64,
        "min_cluster_size": 6,
        "embedding_batch_size": 16,
        "cpu_fallback_on_oom": True
    },
    "sam": {"download": False},
    "streamlit": {"enabled": False, "port": 8501},
    "run": {
        "det_lora_id": "det_lora_01",
        "run_det_fullft": True,
        "seg_lora_id": "seg_lora_01",
        "run_seg_fullft": True
    }
}

print("Configuração carregada:", PARAMS["training"])


## 2. Preparar Colab, Drive, GitHub e YOLO-Master


In [ ]:
import os, sys, subprocess, shutil, importlib.util
from google.colab import drive, userdata

subprocess.run(["nvidia-smi"], check=False)
drive.mount("/content/drive")

RUNTIME = Path(PARAMS["paths"]["drive_runtime"])
for name in ["data", "workspace", "exports", "weights"]:
    (RUNTIME / name).mkdir(parents=True, exist_ok=True)

REPO = Path(PARAMS["paths"]["repo"])
token = userdata.get(PARAMS["secrets"]["github"])
if not token:
    raise RuntimeError("GITHUB_TOKEN não encontrado em Secrets.")

netrc = Path.home() / ".netrc"
netrc.write_text(f"machine github.com\nlogin x-access-token\npassword {token}\n", encoding="utf-8")
netrc.chmod(0o600)
try:
    if REPO.exists():
        subprocess.run(["git", "-C", str(REPO), "pull", "--ff-only"], check=True)
    else:
        subprocess.run(["git", "clone", "https://github.com/ferreis/TCC_Code_Test.git", str(REPO)], check=True)
finally:
    netrc.unlink(missing_ok=True)

os.chdir(REPO)
subprocess.run([sys.executable, "-m", "pip", "install", "--disable-pip-version-check", "-r", "requirements-colab.txt"], check=True)
subprocess.run([sys.executable, "scripts/setup_yolo_master.py", "--dir", PARAMS["paths"]["yolo_master"]], check=True)

check = 'import importlib.util, ultralytics; print(ultralytics.__version__); assert importlib.util.find_spec("ultralytics.utils.lora")'
subprocess.run([sys.executable, "-c", check], check=True)

import torch, ultralytics
print("PyTorch:", torch.__version__, "| CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    print("GPU:", p.name, "| VRAM:", round(p.total_memory / 1024**3, 2), "GB")
print("YOLO-Master:", ultralytics.__version__, "| LoRA:", importlib.util.find_spec("ultralytics.utils.lora") is not None)


## 3. Persistência e datasets Roboflow

O dataset fica persistente no Drive. Durante o treino, `run_stage()` pode copiar apenas o dataset usado para `/content` (SSD local).


In [ ]:
for name in ["data", "workspace", "exports", "weights"]:
    target, link = RUNTIME / name, REPO / name
    if link.is_symlink():
        link.unlink()
    elif link.exists():
        if link.is_dir():
            shutil.rmtree(link)
        else:
            link.unlink()
    link.symlink_to(target, target_is_directory=True)

Path(PARAMS["paths"]["local_data_root"]).mkdir(parents=True, exist_ok=True)

rf_key = userdata.get(PARAMS["secrets"]["roboflow"])
if not rf_key:
    raise RuntimeError("ROBOFLOW_API_KEY não encontrado em Secrets.")
os.environ["ROBOFLOW_API_KEY"] = rf_key

def download_rf(spec):
    if not spec.get("download", False):
        return
    project = spec["project"]
    if str(project).startswith("COLOQUE_"):
        raise ValueError(f"Preencha o slug Roboflow: {project}")
    cmd = [sys.executable, "scripts/download_roboflow.py", "--workspace", PARAMS["roboflow"]["workspace"], "--project", project, "--version", str(spec.get("version", 1)), "--format", PARAMS["roboflow"]["format"], "--output", spec["output"]]
    if spec.get("overwrite", False):
        cmd.append("--overwrite")
    subprocess.run(cmd, check=True)

download_rf(PARAMS["roboflow"]["general"])
for stage_id, spec in PARAMS["roboflow"]["lora"].items():
    if spec.get("download", False):
        print("Baixando:", stage_id)
        download_rf(spec)


## 4. SAM / clustering / Streamlit — opcionais


In [ ]:
if PARAMS["sam"]["download"]:
    subprocess.run([sys.executable, "scripts/download_sam.py"], check=True)

c = PARAMS["clustering"]
if c["enabled"]:
    from tcc_discovery.box_clustering import BoxClusteringConfig, run_box_clustering
    cfg = BoxClusteringConfig(context_scale=c["context_scale"], min_crop_size_px=c["min_crop_size_px"], min_cluster_size=c["min_cluster_size"], embedding_batch_size=c["embedding_batch_size"], dino_cpu_fallback_on_oom=c["cpu_fallback_on_oom"])
    if not (Path(c["output"]) / "catalog.json").exists():
        run_box_clustering(c["source"], c["output"], cfg, overwrite=False)

if PARAMS["streamlit"]["enabled"]:
    from pyngrok import ngrok
    import time
    ngrok_token = userdata.get(PARAMS["secrets"]["ngrok"])
    if not ngrok_token:
        raise RuntimeError("NGROK_AUTHTOKEN não encontrado.")
    ngrok.kill(); ngrok.set_auth_token(ngrok_token)
    streamlit_process = subprocess.Popen(["streamlit", "run", "app.py", "--server.port", str(PARAMS["streamlit"]["port"]), "--server.headless", "true"], stdout=subprocess.DEVNULL, stderr=subprocess.STDOUT)
    time.sleep(3)
    print("Streamlit:", ngrok.connect(PARAMS["streamlit"]["port"], bind_tls=True).public_url)


## 5. Helper de treinamento

Aplica `PARAMS` ao YAML de runtime e faz staging do dataset em `/content` quando habilitado.


In [ ]:
import yaml

if "streamlit_process" in globals() and streamlit_process.poll() is None:
    streamlit_process.terminate()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

PLAN = Path("configs/training_experiments.yaml")
RUNTIME_PLAN = Path("workspace/training_plan_runtime.yaml")
GROUPS = ["smoke_tests", "general_models", "lora_specialists", "full_finetune_controls"]

def run_stage(stage_id, resume=False, dry_run=False):
    cfg = yaml.safe_load(PLAN.read_text(encoding="utf-8"))
    g, t = cfg["global"], PARAMS["training"]
    for k in ["epochs","imgsz","batch","workers","device","amp","optimizer","patience","seed","deterministic","cache","close_mosaic","save_period","plots","verbose"]:
        g[k] = t[k]
    g["project"] = PARAMS["paths"]["training_project"]
    g["eval_split"] = t["eval_split"]

    found = None
    for group in GROUPS:
        for key, item in (cfg.get(group, {}) or {}).items():
            item["enabled"] = key == stage_id
            if key == stage_id:
                found = item
    if found is None:
        raise KeyError(stage_id)

    if stage_id in {"det_general", "smoke_det_yolo26n"}:
        found["model"] = t["detect_model"]
        data_yaml = Path(PARAMS["roboflow"]["general"]["output"]) / "data.yaml"
    elif stage_id in {"seg_general", "smoke_seg_yolo26n"}:
        found["model"] = t["segment_model"]
        data_yaml = Path(found["data"])
    else:
        lora_id = stage_id.replace("det_fullft_", "det_lora_") if stage_id.startswith("det_fullft_") else stage_id
        spec = PARAMS["roboflow"]["lora"].get(lora_id)
        data_yaml = Path(spec["output"]) / "data.yaml" if spec else Path(found["data"])

    data_yaml = (REPO / data_yaml).resolve() if not data_yaml.is_absolute() else data_yaml
    if t["use_local_dataset_cache"] and data_yaml.exists():
        local = Path(PARAMS["paths"]["local_data_root"]) / stage_id
        if t["refresh_local_dataset_cache"] and local.exists():
            shutil.rmtree(local)
        if not (local / "data.yaml").exists():
            shutil.copytree(data_yaml.parent, local, dirs_exist_ok=True)
        data_yaml = local / "data.yaml"
    found["data"] = str(data_yaml)

    if stage_id.startswith("smoke_"):
        found.setdefault("train_args", {}).update(PARAMS["smoke"])
    if "_lora_" in stage_id:
        l = PARAMS["lora"]
        found.setdefault("train_args", {}).update({"lora_r": l["r"], "lora_alpha": l["alpha"], "lora_dropout": l["dropout"], "lora_gradient_checkpointing": l["gradient_checkpointing"], "lora_include_head": l["include_head"]})

    RUNTIME_PLAN.write_text(yaml.safe_dump(cfg, sort_keys=False, allow_unicode=True), encoding="utf-8")
    print("ETAPA:", stage_id, "| DATA:", found["data"], "| BATCH:", found.get("train_args",{}).get("batch",g["batch"]), "| IMGSZ:", found.get("train_args",{}).get("imgsz",g["imgsz"]))
    cmd = [sys.executable, "scripts/train_experiments.py", "--config", str(RUNTIME_PLAN), "--only", stage_id]
    if resume: cmd.append("--resume")
    if dry_run: cmd.append("--dry-run")
    subprocess.run(cmd, check=True)


## 6. Smoke Test — YOLO26n Detecção (3 épocas)


In [ ]:
run_stage("smoke_det_yolo26n")


## 7. Modelo Geral — Detecção


In [ ]:
run_stage("det_general")


## 8. LoRA Especialista + controle Full Fine-Tune

Escolha a LoRA em `PARAMS["run"]["det_lora_id"]`.


In [ ]:
DET_LORA_ID = PARAMS["run"]["det_lora_id"]
run_stage(DET_LORA_ID)
if PARAMS["run"]["run_det_fullft"]:
    run_stage(DET_LORA_ID.replace("det_lora_", "det_fullft_"))


## 9. Segmentação — quando os datasets estiverem prontos


In [ ]:
if Path("data/segmentation/data.yaml").exists():
    run_stage("seg_general")
    SEG_LORA_ID = PARAMS["run"]["seg_lora_id"]
    if Path(f"data/specialists/segmentation/{SEG_LORA_ID.replace('seg_', '')}/data.yaml").exists():
        run_stage(SEG_LORA_ID)
        if PARAMS["run"]["run_seg_fullft"]:
            run_stage(SEG_LORA_ID.replace("seg_lora_", "seg_fullft_"))
else:
    print("Dataset de segmentação ainda não disponível.")


## 10. Resultados e pacote para TCC_Analise


In [ ]:
import pandas as pd

summary = Path(PARAMS["paths"]["training_project"]) / "experiment_summary.csv"
if summary.exists():
    display(pd.read_csv(summary))

subprocess.run([sys.executable, "scripts/export_analysis_bundle.py", "--config", str(RUNTIME_PLAN if RUNTIME_PLAN.exists() else PLAN), "--runs-root", PARAMS["paths"]["training_project"], "--output", PARAMS["paths"]["analysis_export"]], check=True)
print("Baixe:", PARAMS["paths"]["analysis_export"] + ".zip")


## Diagnóstico de GPU

Durante o treino, execute `!nvidia-smi` em outra célula e observe **GPU-Util**. VRAM ocupada não é percentual de processamento.
